# LeetCode 3333 — Find the Original Typed String II bằng Python

- Link: https://leetcode.com/problems/find-the-original-typed-string-ii/
- Độ khó: **Hard**
- Chủ đề: **Đếm phần bù** (tổng số chuỗi gốc − số chuỗi gốc quá ngắn), **DP knapsack bị chặn** với **tổng tiền tố** `O(g · k)`, nhận xét **nếu số nhóm ≥ k thì không cần trừ**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def possibleStringCount(self, word: str, k: int) -> int:
```

## 1. Mục tiêu học

1. Tách `word` thành các **nhóm ký tự liên tiếp giống nhau** (run) độ dài `L_1, …, L_g`. Chuỗi gốc giữ lại `1..L_i` ký tự của nhóm `i` (không được xóa hết một nhóm).
2. Tổng số chuỗi gốc là `∏ L_i`; cần loại các chuỗi có độ dài `< k`.
3. Đếm số cách chọn `c_i ∈ [1, L_i]` với `Σ c_i < k` bằng DP knapsack **bị chặn**; dùng **tổng tiền tố** để mỗi chuyển trạng thái `O(1)` ⇒ `O(g · k)`.
4. Nhận xét cắt: nếu `g ≥ k` thì mọi chuỗi gốc đều dài `≥ g ≥ k` ⇒ đáp án `= ∏ L_i`.
5. Kiểm chứng bằng vét cạn mọi bộ `(c_i)`.

## 2. Đề bài & ví dụ

Alice gõ một chuỗi, nhưng có thể **giữ phím quá lâu** khiến một ký tự bị lặp nhiều lần. Cho chuỗi hiển thị `word` và `k`: đếm số chuỗi gốc **có thể** mà Alice định gõ, biết chuỗi gốc có độ dài **ít nhất `k`**. Modulo `10^9 + 7`.

**Ví dụ 1:** `word = "aabbccdd"`, `k = 7` → `5`.
**Ví dụ 2:** `word = "aabbccdd"`, `k = 8` → `1`.
**Ví dụ 3:** `word = "aaabbb"`, `k = 3` → `8`.

**Ràng buộc:** `1 ≤ len(word) ≤ 5·10^5`; `1 ≤ k ≤ 2000`; chữ thường.

> Mỗi nhóm có thể bị kéo dài **tùy ý** (đây là bản II — bản I chỉ cho phép tối đa một nhóm bị lặp).

In [1]:
from typing import List
from itertools import groupby, product
MOD = 10 ** 9 + 7

examples = [("aabbccdd", 7, 5), ("aabbccdd", 8, 1), ("aaabbb", 3, 8)]
for w, k, exp in examples:
    print(repr(w), 'k =', k, '-> kỳ vọng', exp)

'aabbccdd' k = 7 -> kỳ vọng 5
'aabbccdd' k = 8 -> kỳ vọng 1
'aaabbb' k = 3 -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1. Tách nhóm (run-length)

In [2]:
def runs(word):
    return [len(list(g)) for _, g in groupby(word)]

print(runs("aabbccdd"), runs("aaabbb"), runs("abaa"))

[2, 2, 2, 2] [3, 3] [1, 1, 2]


Lưu ý `"abaa"` có **ba** nhóm `a | b | aa` — hai chữ `a` không liền nhau là hai nhóm khác nhau.

### 3.2. Đếm phần bù

`#(độ dài ≥ k) = ∏ L_i − #(độ dài < k)`. Phần "độ dài `< k`" chỉ liên quan tới tổng `< k ≤ 2000` — nhỏ, trong khi tổng số chuỗi là tích khổng lồ.

### 3.3. Knapsack bị chặn với tổng tiền tố

`dp_i[s]` = số cách chọn `c_1..c_i` (mỗi `c_j ∈ [1, L_j]`) có tổng `s`. Chuyển: `dp_i[s] = Σ_{c=1}^{L_i} dp_{i−1}[s − c]` — một **cửa sổ** liên tiếp ⇒ dùng mảng tổng tiền tố `P[s] = Σ_{t < s} dp_{i−1}[t]`: `dp_i[s] = P[s] − P[max(0, s − L_i)]`.

In [3]:
def window_step(dp, L, K):
    P = [0] * (K + 1)
    for s in range(K):
        P[s + 1] = (P[s] + dp[s]) % MOD
    return [(P[s] - P[max(0, s - L)]) % MOD for s in range(K)]

dp = [1, 0, 0, 0, 0, 0]
dp = window_step(dp, 2, 6); print('sau nhóm L=2:', dp)
dp = window_step(dp, 3, 6); print('sau nhóm L=3:', dp, '(tổng 2..5 với c1∈[1,2], c2∈[1,3])')

sau nhóm L=2: [0, 1, 1, 0, 0, 0]
sau nhóm L=3: [0, 0, 1, 2, 2, 1] (tổng 2..5 với c1∈[1,2], c2∈[1,3])


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: nhóm `[2, 2, 2, 2]`, tổng `∏ = 16` chuỗi gốc. Độ dài chuỗi gốc `= Σ c_i ∈ [4, 8]`. Cần `≥ 7`:

- Độ dài 8: chỉ 1 cách (giữ hết).
- Độ dài 7: chọn một nhóm bỏ 1 ký tự: 4 cách.

⇒ `5` ✓. Theo phần bù: số cách độ dài `< 7` (tức 4, 5, 6) là `1 + 4 + 6 = 11` ⇒ `16 − 11 = 5` ✓.

## 5. Cách 1 — Brute Force: mọi bộ `(c_1, …, c_g)`

In [4]:
def brute(word, k):
    Ls = runs(word)
    return sum(1 for cs in product(*[range(1, L + 1) for L in Ls]) if sum(cs) >= k) % MOD

for w, k, exp in examples:
    print(brute(w, k), '| kỳ vọng', exp)

5 | kỳ vọng 5
1 | kỳ vọng 1
8 | kỳ vọng 8


**Độ phức tạp:** `O(∏ L_i · g)` — chỉ cho chuỗi ngắn.

## 6. Ý tưởng thuật toán / bước đột phá

1. `Ls = runs(word)`, `g = len(Ls)`, `total = ∏ L_i mod p`.
2. Nếu `g ≥ k` ⇒ trả `total`.
3. DP trên tổng `s ∈ [0, k − 1]` qua `g` nhóm với tổng tiền tố: `O(g · k) ≤ 2000 · 2000 = 4·10^6`.
4. Đáp án `= total − Σ_{s < k} dp_g[s]`.

Vì `g < k ≤ 2000` khi phải làm DP, chi phí luôn `≤ 4·10^6` phép tính.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP knapsack trực tiếp `O(g · k · L)`

In [5]:
def dp_naive(word, k):
    Ls = runs(word)
    total = 1
    for L in Ls:
        total = total * L % MOD
    if len(Ls) >= k:
        return total
    dp = [1] + [0] * (k - 1)
    for L in Ls:
        nd = [0] * k
        for s in range(k):
            if dp[s]:
                for c in range(1, L + 1):
                    if s + c >= k:
                        break
                    nd[s + c] = (nd[s + c] + dp[s]) % MOD
        dp = nd
    return (total - sum(dp)) % MOD

for w, k, exp in examples:
    print(dp_naive(w, k), '| kỳ vọng', exp)

5 | kỳ vọng 5
1 | kỳ vọng 1
8 | kỳ vọng 8


### 7.2. Cách 3 — DP với tổng tiền tố (lời giải chính)

In [6]:
def dp_prefix(word, k):
    Ls = runs(word)
    total = 1
    for L in Ls:
        total = total * L % MOD
    if len(Ls) >= k:
        return total
    dp = [1] + [0] * (k - 1)
    for L in Ls:
        dp = window_step(dp, L, k)
    return (total - sum(dp)) % MOD

for w, k, exp in examples:
    print(dp_prefix(w, k), '| kỳ vọng', exp)

5 | kỳ vọng 5
1 | kỳ vọng 1
8 | kỳ vọng 8


### 7.3. Cách 4 — Đếm trực tiếp "độ dài ≥ k" bằng cách dời gốc

Đặt `d_i = c_i − 1 ∈ [0, L_i − 1]` (số ký tự **thêm** ngoài 1 ký tự bắt buộc). Độ dài `= g + Σ d_i`. Điều kiện `< k` ⇔ `Σ d_i < k − g`. DP trên `Σ d_i ∈ [0, k − g − 1]` — mảng ngắn hơn, nhất là khi `g` gần `k`.

In [7]:
def dp_shifted(word, k):
    Ls = runs(word)
    g = len(Ls)
    total = 1
    for L in Ls:
        total = total * L % MOD
    K = k - g                      # cần đếm Σ d_i < K
    if K <= 0:
        return total
    dp = [1] + [0] * (K - 1)
    for L in Ls:
        P = [0] * (K + 1)
        for s in range(K):
            P[s + 1] = (P[s] + dp[s]) % MOD
        dp = [(P[s + 1] - P[max(0, s + 1 - L)]) % MOD for s in range(K)]   # d ∈ [0, L-1]
    return (total - sum(dp)) % MOD

for w, k, exp in examples:
    print(dp_shifted(w, k), '| kỳ vọng', exp)

5 | kỳ vọng 5
1 | kỳ vọng 1
8 | kỳ vọng 8


## 8. TRACE từng bước

Trace cách 3 trên `"aabbbc"`, `k = 5` (nhóm `[2, 3, 1]`, `∏ = 6`).

In [8]:
def trace(word, k):
    Ls = runs(word)
    total = 1
    for L in Ls:
        total *= L
    print('nhóm:', Ls, '| tổng số chuỗi gốc:', total)
    dp = [1] + [0] * (k - 1)
    print('  dp ban đầu (tổng 0..k-1):', dp)
    for L in Ls:
        dp = window_step(dp, L, k)
        print(f'  sau nhóm L = {L}:', dp)
    short = sum(dp)
    print('số chuỗi gốc độ dài < k:', short, '-> đáp án', total - short, '| vét cạn:', brute(word, k))

trace("aabbbc", 5)

nhóm: [2, 3, 1] | tổng số chuỗi gốc: 6
  dp ban đầu (tổng 0..k-1): [1, 0, 0, 0, 0]
  sau nhóm L = 2: [0, 1, 1, 0, 0]
  sau nhóm L = 3: [0, 0, 1, 2, 2]
  sau nhóm L = 1: [0, 0, 0, 1, 2]
số chuỗi gốc độ dài < k: 3 -> đáp án 3 | vét cạn: 3


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def possibleStringCount(self, word: str, k: int) -> int:
        MOD = 10 ** 9 + 7
        Ls = []
        cnt = 1
        for i in range(1, len(word)):
            if word[i] == word[i - 1]:
                cnt += 1
            else:
                Ls.append(cnt)
                cnt = 1
        Ls.append(cnt)
        total = 1
        for L in Ls:
            total = total * L % MOD
        g = len(Ls)
        if g >= k:
            return total
        K = k - g                                    # đếm số bộ d_i ∈ [0, L_i - 1] có Σ d_i < K
        dp = [0] * K
        dp[0] = 1
        for L in Ls:
            P = [0] * (K + 1)
            acc = 0
            for s in range(K):
                acc += dp[s]
                P[s + 1] = acc % MOD
            if L >= K:
                dp = P[1:]                           # cửa sổ phủ hết tiền tố
            else:
                dp = [(P[s + 1] - P[s + 1 - L]) % MOD if s + 1 > L else P[s + 1] for s in range(K)]
        return (total - sum(dp)) % MOD

for w, k, exp in examples:
    got = Solution().possibleStringCount(w, k)
    print(got, 'OK' if got == exp else 'SAI')

5 OK
1 OK
8 OK


## 10. Vì sao thuật toán đúng

1. **Song ánh:** chuỗi gốc khả dĩ ⇔ bộ `(c_1, …, c_g)` với `1 ≤ c_i ≤ L_i`: chuỗi gốc phải có đúng cùng dãy nhóm ký tự (giữ phím không tạo ký tự mới, không gộp hai nhóm khác chữ), và mỗi nhóm còn ít nhất 1 ký tự. Hai bộ khác nhau cho hai chuỗi khác nhau.
2. **Phần bù:** số bộ có `Σ c_i ≥ k` = `∏ L_i` − số bộ có `Σ c_i < k`.
3. **DP:** `dp_i[s]` đếm đúng số bộ trên `i` nhóm đầu có tổng `s` (quy nạp; mỗi bộ có đúng một `c_i` cuối). Tổng tiền tố chỉ là cách cộng cửa sổ nhanh.
4. **Cắt `g ≥ k`:** mọi bộ có `Σ c_i ≥ g ≥ k`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(∏ L_i · g)` | `O(g)` |
| 2. DP trực tiếp | `O(n + g · k · min(L, k))` | `O(k)` |
| 3. DP tổng tiền tố | `O(n + g · k)` | `O(k)` |
| 4. DP dời gốc | `O(n + g · (k − g))` | `O(k − g)` |

`n = len(word)` cho bước tách nhóm.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'dp_naive': dp_naive, 'dp_prefix': dp_prefix, 'dp_shifted': dp_shifted,
         'submit': lambda w, k: Solution().possibleStringCount(w, k)}
edge = [("a", 1, 1), ("aaaa", 1, 4), ("aaaa", 4, 1), ("aaaa", 5, 0), ("abc", 2, 1), ("abaa", 1, 2), ("aabb", 3, 3)]
for w, k, exp in examples + edge:
    assert brute(w, k) == exp, (w, k, brute(w, k))
    for name, f in impls.items():
        assert f(w, k) == exp, (name, w, k, f(w, k))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Ca `"aaaa"`, `k = 5`: chuỗi gốc dài tối đa 4 ⇒ `0` — công thức phần bù vẫn đúng (`4 − 4 = 0`).

### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3333)
cnt = 0
for _ in range(600):
    w = ''.join(rng.choice('ab' if rng.random() < 0.5 else 'abc') * rng.randint(1, 4) for _ in range(rng.randint(1, 5)))
    k = rng.randint(1, len(w) + 2)
    exp = brute(w, k)
    for name, f in impls.items():
        assert f(w, k) == exp, (name, w, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test.')

Khớp vét cạn trên 600 bộ test.


### 12.3. Ca lớn `len(word) = 5·10^5`, `k = 2000`

In [12]:
import time
def make_word(groups, maxlen):
    out, prev = [], ''
    for _ in range(groups):
        c = rng.choice([ch for ch in 'abcdefghij' if ch != prev]); prev = c
        out.append(c * rng.randint(1, maxlen))
    return ''.join(out)

big = {
    '1999 nhóm dài': (make_word(1999, 250)[:5 * 10 ** 5], 2000),
    '1000 nhóm, k=2000': (make_word(1000, 500), 2000),
    'một nhóm 5e5': ('a' * 5 * 10 ** 5, 2000),
    'nhiều nhóm (g >= k)': (make_word(10 ** 5, 5), 2000),
}
for name, (w, k) in big.items():
    t = time.time(); a = Solution().possibleStringCount(w, k); t1 = time.time() - t
    t = time.time(); b = dp_prefix(w, k); t2 = time.time() - t
    assert a == b
    print(f"{name:22s}: n = {len(w):6d}, g = {len(runs(w)):6d} -> {a:10d} | nộp {t1:.2f}s | cách 3 {t2:.2f}s")

1999 nhóm dài         : n = 248735, g =   1999 ->  503344699 | nộp 0.01s | cách 3 0.68s


1000 nhóm, k=2000     : n = 243348, g =   1000 ->  127502898 | nộp 0.17s | cách 3 0.34s
một nhóm 5e5          : n = 500000, g =      1 ->     498001 | nộp 0.03s | cách 3 0.01s
nhiều nhóm (g >= k)   : n = 300022, g = 100000 ->  669061636 | nộp 0.02s | cách 3 0.02s


Kiểm tra độc lập: một nhóm duy nhất dài `L` ⇒ đáp án `= max(0, L − k + 1)` (chọn `c ∈ [k, L]`).

In [13]:
assert Solution().possibleStringCount('a' * 5 * 10 ** 5, 2000) == 5 * 10 ** 5 - 2000 + 1
assert Solution().possibleStringCount('z' * 10, 2000) == 0
print('một nhóm: khớp L - k + 1')

một nhóm: khớp L - k + 1


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép xóa hết một nhóm (`c_i ∈ [0, L_i]`)

In [14]:
def wrong_allow_zero(word, k):
    Ls = runs(word)
    return sum(1 for cs in product(*[range(0, L + 1) for L in Ls]) if sum(cs) >= k) % MOD   # SAI

print('"aab", k = 1 -> sai :', wrong_allow_zero("aab", 1), '| đúng:', Solution().possibleStringCount("aab", 1))

"aab", k = 1 -> sai : 5 | đúng: 2


**Vì sao sai:** giữ phím chỉ làm **thêm** ký tự; mỗi nhóm trên màn hình có ít nhất một lần gõ thật. Với `"aab"`: đúng là `{"aab", "ab"}`; bản sai đếm thêm `"aa"`, `"a"`, `"b"` (xóa hẳn nhóm).

### Lỗi 2 — Trừ các chuỗi độ dài `≤ k` thay vì `< k`

In [15]:
def wrong_leq(word, k):
    Ls = runs(word)
    total = 1
    for L in Ls:
        total = total * L % MOD
    dp = [1] + [0] * k
    for L in Ls:
        dp = window_step(dp, L, k + 1)                 # SAI: đếm cả tổng = k
    return (total - sum(dp)) % MOD

print('sai :', wrong_leq("aabbccdd", 8), '| đúng:', Solution().possibleStringCount("aabbccdd", 8))

sai : 0 | đúng: 1


**Vì sao sai:** "độ dài **ít nhất** `k`" gồm cả độ dài đúng bằng `k` — chuỗi gốc `"aabbccdd"` (dài 8) bị trừ nhầm.

### Lỗi 3 — Gom nhóm theo **chữ cái** thay vì theo **đoạn liên tiếp**

In [16]:
from collections import Counter
def wrong_group_by_letter(word, k):
    Ls = list(Counter(word).values())                  # SAI: "abaa" thành a:3, b:1
    return sum(1 for cs in product(*[range(1, L + 1) for L in Ls]) if sum(cs) >= k) % MOD

print('sai :', wrong_group_by_letter("abaa", 1), '| đúng:', Solution().possibleStringCount("abaa", 1))

sai : 3 | đúng: 2


**Vì sao sai:** chữ `a` đầu tiên đứng một mình (đã bị `b` chen giữa) nên không thể bị "kéo dài" thành phần của nhóm `aa` cuối.

### Lỗi 4 — Cửa sổ tổng tiền tố lệch một (cho phép `c ∈ [0, L − 1]`)

In [17]:
def wrong_window(word, k):
    Ls = runs(word)
    total = 1
    for L in Ls:
        total = total * L % MOD
    if len(Ls) >= k:
        return total
    dp = [1] + [0] * (k - 1)
    for L in Ls:
        P = [0] * (k + 1)
        for s in range(k):
            P[s + 1] = (P[s] + dp[s]) % MOD
        dp = [(P[s + 1] - P[max(0, s + 1 - L)]) % MOD for s in range(k)]    # SAI: cửa sổ [s-L+1, s]
    return (total - sum(dp)) % MOD

print('sai :', wrong_window("aabbccdd", 7), '| đúng:', Solution().possibleStringCount("aabbccdd", 7))

sai : 0 | đúng: 5


**Vì sao sai:** cửa sổ `[s − L + 1, s]` tương ứng `c ∈ [0, L − 1]` — đúng cho biến **dời gốc** `d_i` (cách 4) nhưng sai cho `c_i` nếu mảng DP không dời. Khi dùng tổng tiền tố phải viết rõ khoảng `c` đang cộng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số cấu hình có tổng ≥ k, tổng có thể rất lớn" ⇒ **phần bù**: tổng số − số cấu hình tổng `< k`.
- Knapsack mà mỗi món có số lượng chọn trong một **đoạn liên tiếp** ⇒ tổng tiền tố (hoặc cửa sổ trượt).
- Ràng buộc `k ≤ 2000` nhỏ hơn hẳn `n ≤ 5·10^5` ⇒ DP theo `k`, kèm cắt `g ≥ k`.

## 15. Biến thể của bài

1. **LC 3330 (bản I):** tối đa **một** nhóm bị kéo dài ⇒ `1 + Σ (L_i − 1)`.
2. Độ dài chuỗi gốc **đúng bằng** `k` ⇒ `dp_g[k]` (DP tới `k`).
3. Độ dài trong đoạn `[a, b]` ⇒ `F(a) − F(b + 1)` với `F(x)` = số chuỗi độ dài `≥ x`.
4. Mỗi nhóm có giới hạn kéo dài **riêng** (`c_i ≥ L_i − r_i`) ⇒ đổi cận cửa sổ.

In [18]:
# Biến thể 1, 2, 3
def lc3330(word):
    return 1 + sum(L - 1 for L in runs(word))

def exact_length(word, k):
    Ls = runs(word)
    dp = [1] + [0] * k
    for L in Ls:
        dp = window_step(dp, L, k + 1)
    return dp[k]

def length_in_range(word, a, b):
    F = lambda x: Solution().possibleStringCount(word, x) if x >= 1 else Solution().possibleStringCount(word, 1)
    return (F(a) - F(b + 1)) % MOD

rng = random.Random(15)
for _ in range(300):
    w = ''.join(rng.choice('abc') * rng.randint(1, 4) for _ in range(rng.randint(1, 5)))
    Ls = runs(w)
    combos = list(product(*[range(1, L + 1) for L in Ls]))
    assert lc3330(w) == 1 + sum(1 for cs in combos if sum(1 for c, L in zip(cs, Ls) if c < L) == 1)
    k = rng.randint(1, len(w) + 1)
    assert exact_length(w, k) == sum(1 for cs in combos if sum(cs) == k)
    a = rng.randint(1, len(w)); b = rng.randint(a, len(w) + 1)
    assert length_in_range(w, a, b) == sum(1 for cs in combos if a <= sum(cs) <= b)
print('Biến thể 1, 2, 3 khớp vét cạn.')

Biến thể 1, 2, 3 khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi bộ `c_i` | `O(∏ L_i)` | Oracle |
| 2 | Knapsack trực tiếp | `O(g k L)` | Chậm khi nhóm dài |
| 3 | Knapsack + tổng tiền tố | `O(g k)` | Chuẩn |
| 4 | Dời gốc `d_i = c_i − 1` | `O(g (k − g))` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `word = "abc"`, `k = 4`.

**Bài 2 (tính tay):** `word = "aaabb"`, `k = 4`.

**Bài 3 (điền TODO):** trả về **phân phối** số chuỗi gốc theo độ dài (từ `g` tới `n`) cho chuỗi ngắn.

**Bài 4 (tự viết lại):** biến thể 4 — mỗi nhóm `i` được xóa **tối đa** `r_i` ký tự.

**Bài 5 (mini challenge):** liệt kê (theo thứ tự từ điển) mọi chuỗi gốc khi tổng số `≤ 1000`.

In [19]:
def length_distribution_todo(word):
    # TODO
    raise NotImplementedError

def limited_deletions_todo(word, k, r):
    # TODO
    raise NotImplementedError

def list_originals_todo(word, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mỗi nhóm dài 1. (2) Chỉ một chuỗi gốc dài 3. (3) `0`.

**Bài 2.** (1) Nhóm `[3, 2]`, tổng 6 chuỗi. (2) Độ dài `< 4`: `(1,1), (1,2), (2,1)`. (3) `6 − 3 = 3`.

**Bài 3.** (1) DP tới tổng `n`. (2) `dp[s]` với `s ∈ [g, n]`. (3) Tổng bằng `∏ L_i`.

**Bài 4.** (1) `c_i ∈ [L_i − r_i, L_i]` (và `≥ 1`). (2) Cửa sổ `[s − L_i, s − max(1, L_i − r_i)]`. (3) Vét cạn.

**Bài 5.** (1) Duyệt tích Descartes theo thứ tự `c_i`. (2) Ghép chuỗi `ch_i * c_i`. (3) Sắp xếp kết quả.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().possibleStringCount("abc", 4) == 0
assert Solution().possibleStringCount("aaabb", 4) == 3 == brute("aaabb", 4)

def length_distribution(word):
    Ls = runs(word); n = len(word)
    dp = [1] + [0] * n
    for L in Ls:
        dp = window_step(dp, L, n + 1)
    return {s: dp[s] for s in range(n + 1) if dp[s]}

dist = length_distribution("aabbccdd")
assert sum(dist.values()) == 16 and dist[7] == 4 and dist[8] == 1
print('Bài 1: 0 | Bài 2: 3 | Bài 3:', dist)

Bài 1: 0 | Bài 2: 3 | Bài 3: {4: 1, 5: 4, 6: 6, 7: 4, 8: 1}


In [21]:
# Bài 4, 5
def limited_deletions(word, k, r):
    Ls = runs(word)
    dp = [1] + [0] * (len(word))
    K = len(word) + 1
    for L, ri in zip(Ls, r):
        lo = max(1, L - ri)
        P = [0] * (K + 1)
        for s in range(K):
            P[s + 1] = (P[s] + dp[s]) % MOD
        dp = [(P[max(0, s - lo + 1)] - P[max(0, s - L)]) % MOD for s in range(K)]   # c ∈ [lo, L]
    return sum(dp[s] for s in range(k, K)) % MOD

def list_originals(word, k):
    groups = [(ch, len(list(g))) for ch, g in groupby(word)]
    out = [''.join(ch * c for (ch, _), c in zip(groups, cs))
           for cs in product(*[range(1, L + 1) for _, L in groups])]
    return sorted(s for s in out if len(s) >= k)

rng = random.Random(4)
for _ in range(300):
    w = ''.join(rng.choice('abc') * rng.randint(1, 4) for _ in range(rng.randint(1, 5)))
    Ls = runs(w); r = [rng.randint(0, L) for L in Ls]; k = rng.randint(1, len(w) + 1)
    exp = sum(1 for cs in product(*[range(max(1, L - ri), L + 1) for L, ri in zip(Ls, r)]) if sum(cs) >= k)
    assert limited_deletions(w, k, r) == exp
    lst = list_originals(w, k)
    assert len(lst) == len(set(lst)) == brute(w, k)
print('Bài 4, 5 khớp vét cạn. Ví dụ:', list_originals("aabbccdd", 7))

Bài 4, 5 khớp vét cạn. Ví dụ: ['aabbccd', 'aabbccdd', 'aabbcdd', 'aabccdd', 'abbccdd']


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
Ls = độ dài các run; total = ∏ Ls; g = len(Ls)
if g >= k: return total
K = k - g; dp = [1] + [0]*(K-1)               # d_i = c_i - 1 ∈ [0, L_i - 1]
for L in Ls: dp[s] = P[s+1] - P[max(0, s+1-L)]  (P = tiền tố của dp cũ)
return (total - sum(dp)) % MOD
```

**Checklist:**

- [ ] Nhóm theo **đoạn liên tiếp**, không theo chữ cái?
- [ ] Mỗi nhóm giữ ít nhất 1 ký tự?
- [ ] Trừ độ dài **`< k`** (không phải `≤ k`)?
- [ ] Cắt `g ≥ k` để tránh DP vô ích?
- [ ] Cửa sổ tổng tiền tố đúng với biến đang dùng (`c` hay `d`)?